In [9]:
import kagglehub
import shutil
import random
import os

In [10]:
# Download latest version
path = kagglehub.dataset_download("seroshkarim/cotton-leaf-disease-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'cotton-leaf-disease-dataset' dataset.
Path to dataset files: /kaggle/input/cotton-leaf-disease-dataset


In [11]:
print(os.listdir(path))

['cotton']


In [12]:
path = kagglehub.dataset_download("seroshkarim/cotton-leaf-disease-dataset")


data_dir = os.path.join(path, "cotton")

print("Using dataset from:", data_dir)


output_dir = "dataset_split"

train_dir = os.path.join(output_dir, "train")
val_dir   = os.path.join(output_dir, "val")
test_dir  = os.path.join(output_dir, "test")


for split in [train_dir, val_dir, test_dir]:
    os.makedirs(split, exist_ok=True)


classes = os.listdir(data_dir)

print("Classes found:", classes)


for cls in classes:
    class_path = os.path.join(data_dir, cls)

    if not os.path.isdir(class_path):
        continue

    images = os.listdir(class_path)
    random.shuffle(images)

    total = len(images)

    train_end = int(0.6 * total)
    val_end   = int(0.8 * total)

    train_imgs = images[:train_end]
    val_imgs   = images[train_end:val_end]
    test_imgs  = images[val_end:]


    os.makedirs(os.path.join(train_dir, cls), exist_ok=True)
    os.makedirs(os.path.join(val_dir, cls), exist_ok=True)
    os.makedirs(os.path.join(test_dir, cls), exist_ok=True)


    for img in train_imgs:
        shutil.copy(
            os.path.join(class_path, img),
            os.path.join(train_dir, cls, img)
        )


    for img in val_imgs:
        shutil.copy(
            os.path.join(class_path, img),
            os.path.join(val_dir, cls, img)
        )


    for img in test_imgs:
        shutil.copy(
            os.path.join(class_path, img),
            os.path.join(test_dir, cls, img)
        )

print("Dataset split done ")

Using Colab cache for faster access to the 'cotton-leaf-disease-dataset' dataset.
Using dataset from: /kaggle/input/cotton-leaf-disease-dataset/cotton
Classes found: ['fussarium_wilt', 'curl_virus', 'healthy', 'bacterial_blight']
Dataset split done 


In [14]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [21]:
img_size = (224, 224)
batch_size = 32

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_gen = ImageDataGenerator(rescale=1./255)

test_gen = ImageDataGenerator(rescale=1./255)

In [22]:
train_data = train_gen.flow_from_directory(
    "dataset_split/train",
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical'
)

Found 1591 images belonging to 5 classes.


In [23]:
val_data = val_gen.flow_from_directory(
    "dataset_split/val",
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical'
)

Found 827 images belonging to 5 classes.


In [24]:
test_data = test_gen.flow_from_directory(
    "dataset_split/test",
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

Found 828 images belonging to 5 classes.


In [25]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2, EfficientNetB0, ResNet50

def build_model(base_name, num_classes):

    if base_name == "mobilenet":
        base = MobileNetV2(input_shape=(224,224,3),
                            include_top=False,
                            weights='imagenet')

    elif base_name == "efficientnet":
        base = EfficientNetB0(input_shape=(224,224,3),
                              include_top=False,
                              weights='imagenet')

    elif base_name == "resnet":
        base = ResNet50(input_shape=(224,224,3),
                        include_top=False,
                        weights='imagenet')

    base.trainable = False

    model = models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [26]:
def train_model(model, train_data, val_data):
    history = model.fit(
        train_data,
        validation_data=val_data,
        epochs=10,
        verbose=1
    )
    return model

In [27]:
def evaluate_model(model, test_data, name):
    loss, acc = model.evaluate(test_data, verbose=0)
    print(f"{name} -> Test Accuracy: {acc:.4f}")
    return acc

In [28]:
results = {}

# MobileNet
model_mob = build_model("mobilenet", train_data.num_classes)
model_mob = train_model(model_mob, train_data, val_data)
results["MobileNetV2"] = evaluate_model(model_mob, test_data, "MobileNetV2")

# EfficientNet
model_eff = build_model("efficientnet", train_data.num_classes)
model_eff = train_model(model_eff, train_data, val_data)
results["EfficientNetB0"] = evaluate_model(model_eff, test_data, "EfficientNetB0")

# ResNet
model_res = build_model("resnet", train_data.num_classes)
model_res = train_model(model_res, train_data, val_data)
results["ResNet50"] = evaluate_model(model_res, test_data, "ResNet50")

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Epoch 1/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 143s 3s/step - accuracy: 0.7662 - loss: 0.6130 - val_accuracy: 0.9274 - val_loss: 0.2247
Epoch 2/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 121s 2s/step - accuracy: 0.9378 - loss: 0.1984 - val_accuracy: 0.9601 - val_loss: 0.1338
Epoch 3/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 136s 3s/step - accuracy: 0.9635 - loss: 0.1306 - val_accuracy: 0.9819 - val_loss: 0.0865
Epoch 4/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 127s 3s/step - accuracy: 0.9635 - loss: 0.1151 - val_accuracy: 0.9807 - val_loss: 0.0727
Epoch 5/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 135s 3s/step - accuracy: 0.9761 - loss: 0.0797 - val_accuracy: 0.9915 - val_loss: 0.0464
Epoch 6/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 131s 3s/step - accuracy: 0.9799 - loss: 0.0650 - val_accuracy: 0.9915 - val_loss: 0.0347
Epoch 7/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 124s 2s/step - accuracy: 0.9830 - loss: 0.0604 - val_accuracy: 0.9940 - val_loss: 0.0295
Epoch 8/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 127s 3s/step - accuracy:

In [29]:
for k,v in results.items():
    print(k, ":", round(v*100,2), "%")

MobileNetV2 : 99.52 %
EfficientNetB0 : 27.9 %
ResNet50 : 49.52 %


In [30]:
model_mob.save("mobilenet_model.keras")

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model_mob)
tflite_model = converter.convert()

open("model.tflite", "wb").write(tflite_model)

In [33]:
from tensorflow.keras.preprocessing import image
import numpy as np

In [53]:
img_path = "test_image.jpg"

img = image.load_img(img_path, target_size=(224, 224))

img_array = image.img_to_array(img)
img_array = img_array / 255.0

img_array = np.expand_dims(img_array, axis=0)

In [54]:
prediction = model_mob.predict(img_array)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step


In [55]:
class_names = list(train_data.class_indices.keys())

pred_class = class_names[np.argmax(prediction)]
confidence = np.max(prediction)

In [56]:
print("Prediction:", pred_class)
print("Confidence:", confidence)

Prediction: bacterial_blight
Confidence: 0.940881


In [57]:
img_path = "test_image_2.jpg"

img = image.load_img(img_path, target_size=(224, 224))

img_array = image.img_to_array(img)
img_array = img_array / 255.0

img_array = np.expand_dims(img_array, axis=0)

In [58]:
prediction_2 = model_mob.predict(img_array)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step


In [59]:
class_names = list(train_data.class_indices.keys())

pred_class = class_names[np.argmax(prediction_2)]
confidence = np.max(prediction_2)

In [60]:
print("Prediction:", pred_class)
print("Confidence:", confidence)

Prediction: curl_virus
Confidence: 0.9999769


In [62]:
img_path = "test_image_3.jpg"

img = image.load_img(img_path, target_size=(224, 224))

img_array = image.img_to_array(img)
img_array = img_array / 255.0

img_array = np.expand_dims(img_array, axis=0)

prediction_3 = model_mob.predict(img_array)

class_names = list(train_data.class_indices.keys())

pred_class = class_names[np.argmax(prediction_3)]
confidence = np.max(prediction_3)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step


In [63]:
print("Prediction:", pred_class)
print("Confidence:", confidence)

Prediction: curl_virus
Confidence: 0.7243877


In [65]:
img_path = "test_image_4.jpg"

img = image.load_img(img_path, target_size=(224, 224))

img_array = image.img_to_array(img)
img_array = img_array / 255.0

img_array = np.expand_dims(img_array, axis=0)

prediction_4 = model_mob.predict(img_array)

class_names = list(train_data.class_indices.keys())

pred_class = class_names[np.argmax(prediction_4)]
confidence = np.max(prediction_4)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step


In [66]:
print("Prediction:", pred_class)
print("Confidence:", confidence)

Prediction: fussarium_wilt
Confidence: 0.48948815


In [68]:
img_path = "test_image_5.jpg"

img = image.load_img(img_path, target_size=(224, 224))

img_array = image.img_to_array(img)
img_array = img_array / 255.0

img_array = np.expand_dims(img_array, axis=0)

prediction_5 = model_mob.predict(img_array)

class_names = list(train_data.class_indices.keys())

pred_class = class_names[np.argmax(prediction_5)]
confidence = np.max(prediction_5)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step


In [69]:
print("Prediction:", pred_class)
print("Confidence:", confidence)

Prediction: healthy
Confidence: 0.96426314


In [ ]:
label_map = {
    "bacterial_blight": "Bacterial Blight - اللفحة البكتيرية",
    "curl_virus": "Curl Virus - فيروس تجعد الأوراق",
    "fussarium_wilt": "Fusarium Wilt - الذبول الفيوزاريومي",
    "healthy": "Healthy - نبات سليم"
}